# Translation Pseudocode for Each Event
Each event has an associated notebook with a worked example. This notebook serves to string the code together for each event into a pseudocode summary.

## Synopsis for New Flow: 

Each event type has a list of cases in events.yaml. The EventTypeOperator class guides the high-level flow of the code by:  

1. Storing the observation types
2. Storing the metrics for each observation   
3. Holding the forecast data for later subsetting and processing
4. Defining how observations and forecasts interact

The forecast is then lazily loaded from whatever source (zarr or virtualizarr file) is provided. Each observation is loaded aided by an Observation class object, which stores:  

1. The metrics to use with the observation
2. The default and/or derived variables needed 
3. The source of the data
4. The modification code to process the data
5. The output Dataset format.  

>The metric storage in Observation allows for different metrics to be used for certain variables and not others in an EventType. For example, if in an augmented Heatwave event analysis, a user might want MAE for 850mb temps and MBE for 2m temperature.  

For each case in the events, the forecast data is checked out to see if the timestamps exist. If so, the unmodified variables from the forecasts are subset as are the times. At this point, the forecast data also is converted to a (init_time,valid_time,latitude,longitude) format.

Derived variables are now calculated for observation data and forecast data.   

>There are some event types that need derived variables, such as Craven-Brooks Significant Severe (CBSS) or integrated vapor transport and atmospheric river masks. Some observation data will have different derived variables to compare against (e.g. CBSS vs Practically Perfect storm report percentiles), which necessitates a directive built into each EventTypeOperator to clarify what is getting compared, and how. **Not all events use derived variables, but some events have both derived and default variables.**  

The forecast is in a singular format but might need to be transformed into one or more different formats for the event, such as (time, latitude, longitude) or (time, location). For each Observation, this transformation now occurs if needed with code in the Observation object and both the Observation and Forecast is combined into a discrete EvaluationObject that can be used to compute the metrics in that object independently.  

The metrics are then computed in this flow:  

1. EvaluationObjects are looped through
2. Metrics in each EvaluationObject are processed
3. Outputs are appended to a master DataFrame for final output with associated metadata (observation type, lead time, metric name, event type, location, case title)



## Heat Wave

### Current:

**Data Load and Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; generate IndividualHeatWaveCase dataclasses for subsetting  

**Data Subsetting and Processing:**  
&emsp; load forecast from source  
&emsp; load GHCN  
&emsp; load ERA5  
&emsp; subset forecast dataset using metadata  
  
&emsp; for each event_type (just heatwave for now):  
&emsp;&emsp; create the HeatWave event class with IndividualHeatWaveCase objects  

**Individual Case Evaluation:**  
&emsp; for each case in event.cases:  
&emsp;&emsp; create CaseEvaluationData object for ERA5  
&emsp;&emsp; create CaseEvaluationData object for GHCN  
  
&emsp; for each CaseEvaluationData object:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset variable (surface temp)  
&emsp;&emsp; subset time range  
&emsp;&emsp; if point obs:  
&emsp;&emsp;&emsp; filter by case id  
&emsp;&emsp;&emsp; convert longitude to 0-360  
&emsp;&emsp;&emsp; align_point_obs_from_gridded() to match gridpoints  
&emsp;&emsp;&emsp; convert from pandas dataframe to xarray dataset  

**Metric Computation:**  
&emsp; for each data variable (only one for heatwave):  
&emsp;&emsp; for each metric in metric_list:  
&emsp;&emsp;&emsp; instantiate metric()  
&emsp;&emsp;&emsp; for each observation_type in ['gridded', 'point']:  
&emsp;&emsp;&emsp;&emsp; compute metric[data_var]  
&emsp;&emsp;&emsp;&emsp; format result into dataframe

### Refactor:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with heat_wave cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data  

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (surface temp)  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**  
&emsp;&emsp; if variables include a DerivedVariable:  
&emsp;&emsp;&emsp; generate the variable from the Observation and forecast data with DerivedVariable  

&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; transform subset forecast data to Observation coordinates  
&emsp;&emsp;&emsp; combine subset forecast and Observation into EvaluationObject  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  

event <- HeatWave
forecast <- AIFS

cases: [case1, case2] | event
observation_types: [ERA5, GHCN]
metrics: [metric1, metric2, metric3] - applied for all cases
- metric3 can only be computed for ERA5
applied_metrics: [metric1<ERA5>, metric1<GHCN>, metric2<ERA5>, ..., metric3<ERA5>] 
applied_metrics <- cartesian product(metrics, observation_types) |  observation type can be used by metric



for each case:
    case <- from cases
    for each applied_metric:
        metric, obs_type <- from applied_metrics

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data

### Loopified:

event <- HeatWave  
forecast <- AIFS

cases: [case1, case2] | event  
observation_types: [ERA5,GHCN]  
metrics: [Regional RMSE (m1), Highest Minimum Temp MAE (m2), Maximum Temp MAE (m3), Onset ME (m4), Duration ME (m5)]  
  - applied for all cases


applied_metrics <- cartesian product(metrics, observation_types) | iff observation type can be used by metric  
applied_metrics: [m1<ERA5>, m1<GHCN>, m2<ERA5>, m2<GHCN>, m3<ERA5>, m3<GHCN>, m4<ERA5>, m5<ERA5>]

variable_combination: [[surface_temp<forecast>,surface_temp<ERA5>,surface_temp<GHCN>]]  
  - same variable for obs and forecast  
  - applied for all cases  

for each case:  
&emsp;  case <- from cases  
&emsp;&emsp;  for each applied_metric:  
&emsp;&emsp;&emsp;        metric, obs_type <- from applied_metrics  

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data
        5. compute variable_combination for forecast data
        6. compute variable_combination for observation data
        7. merge observation and forecast data
        6. compute applied_metric from observation and applied data

## Freeze

### Current

**Data Load and Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; generate IndividualFreezeCase dataclasses for subsetting  

**Data Subsetting and Processing:**  
&emsp; load forecast from source  
&emsp; load GHCN  
&emsp; load ERA5  
&emsp; subset forecast dataset using metadata  
  
&emsp; for each event_type (just Freeze for now):  
&emsp;&emsp; create the Freeze event class with IndividualFreezeCase objects  

**Individual Case Evaluation:**  
&emsp; for each case in event.cases:  
&emsp;&emsp; create CaseEvaluationData object for ERA5  
&emsp;&emsp; create CaseEvaluationData object for GHCN  
  
&emsp; for each CaseEvaluationData object:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset variable (surface temp)  
&emsp;&emsp; subset time range  
&emsp;&emsp; if point obs:  
&emsp;&emsp;&emsp; filter by case id  
&emsp;&emsp;&emsp; convert longitude to 0-360  
&emsp;&emsp;&emsp; align_point_obs_from_gridded() to match gridpoints  
&emsp;&emsp;&emsp; convert from pandas dataframe to xarray dataset  

**Metric Computation:**  
&emsp; for each data variable (only one for freeze):  
&emsp;&emsp; for each metric in metric_list:  
&emsp;&emsp;&emsp; instantiate metric()  
&emsp;&emsp;&emsp; for each observation_type in ['gridded', 'point']:  
&emsp;&emsp;&emsp;&emsp; compute metric[data_var]  
&emsp;&emsp;&emsp;&emsp; format result into dataframe

### Refactor:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with freeze cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data  

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (surface temp)  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**  
&emsp;&emsp; if variables include a DerivedVariable:  
&emsp;&emsp;&emsp; generate the variable from the Observation and forecast data with DerivedVariable  

&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; transform subset forecast data to Observation coordinates  
&emsp;&emsp;&emsp; combine subset forecast and Observation into EvaluationObject  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  

### Loopified:

event <- Freeze  
forecast <- AIFS

cases: [case1, case2] | event  
observation_types: [ERA5,GHCN]  
metrics: [Regional RMSE (m1), Minimum Temp MAE (m2), Onset ME (m3), Duration ME (m4)]  
  - applied for all cases


applied_metrics <- cartesian product(metrics, observation_types) | iff observation type can be used by metric  
applied_metrics: [m1<ERA5>, m1<GHCN>, m2<ERA5>, m2<GHCN>, m3<ERA5>, m4<ERA5>]

variable_combination: [[surface_temp<forecast>,surface_temp<ERA5>,surface_temp<GHCN>]]  
  - same variable for obs and forecast  
  - applied for all cases  

for each case:  
&emsp;  case <- from cases  
&emsp;&emsp;  for each applied_metric:  
&emsp;&emsp;&emsp;        metric, obs_type <- from applied_metrics  

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data
        5. compute variable_combination for forecast data
        6. compute variable_combination for observation data
        7. merge observation and forecast data
        6. compute applied_metric from observation and applied data

## Severe Convection

### Refactor:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with severe cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data (Local Storm Reports)

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (specific humidity/relative humidity/dewpoint, temperature, u/v winds, geopotential_at_surface) and bounds  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**  
&emsp;&emsp; variables do include a DerivedVariable:   
&emsp;&emsp;&emsp; generate craven brooks sigsvr for the case date range over region, aggregate to max value at each timestep  
&emsp;&emsp;&emsp; generate practically perfect hindcasts using LSR Observations for case using EventTypeOperator bounds

&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; transform subset forecast data to Observation coordinates (not needed here, already in (time,lat,lon))  
&emsp;&emsp;&emsp; combine subset forecast and Observation (on the same grid) into EvaluationObject with scores BinaryContingencyManager and thresholds  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  

### Loopified:

event <- Severe Convection  
forecast <- AIFS

cases: [case1, case2] | event  
observation_types: [LSR]  
metrics: [SC Lead Time Detection (m1), CSI/IOU (m2), FAR (m3), Regional Hits/Misses (m4), Individual Hits/Misses (m5)]  
  - applied for all cases


applied_metrics <- cartesian product(metrics, observation_types) | iff observation type can be used by metric  
applied_metrics: [m1<LSR>, m2<LSR>, m3<LSR>, m4<LSR>, m5<LSR>]

variable_combination: [[CBSS<forecast>,PPH<LSR>,LSR<LSR>] [formation_signal<forecast>]]  
  - different variable for obs and forecast; needs to be derived in forecast and observation separately
  - applied for all cases
  - derived through:  
        1. practically_perfect_hindcast  
        2. craven_sigsvr


for each case:  
&emsp;  case <- from cases  
&emsp;&emsp;  for each applied_metric:  
&emsp;&emsp;&emsp;        metric, obs_type <- from applied_metrics  

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data
        5. compute variable_combination for forecast data
        6. compute variable_combination for observation data
        7. merge observation and forecast data
        6. compute applied_metric from observation and applied data

## Tropical Cyclone

### Refactor:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with tropical_cyclone cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data (IBTrACS, Tomer Burg CSVs?), convert into Dataset(s)  

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (surface temp) and bounds  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**   

&emsp;&emsp; variables do include a DerivedVariable:   
&emsp;&emsp;&emsp; generate IVT, AR mask, and AR/Land overlap for the case date range over region

&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; for each variable in Observation:  
&emsp;&emsp;&emsp;&emsp; transform subset forecast data to Observation coordinates using TC Tracker algorithm directed in Observation code  
&emsp;&emsp;&emsp;&emsp; combine subset forecast and Observation into EvaluationObject  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  

### Loopified:

event <- TC  
forecast <- AIFS

cases: [case1, case2] | event  
observation_types: [IBTrACS, ERA5]  
metrics: [TC Lead Time Detection (m1), Spatial Landfall Error (m2), Temporal Landfall Error (m3), Landfall Intensity MAE (m4)]  
  - applied for all cases


applied_metrics <- cartesian product(metrics, observation_types) | iff observation type can be used by metric  
applied_metrics: [m1<ERA5>, m1<IBTrACS> m2<ERA5>, m2<IBTrACS>, m3<ERA5>, m3<IBTrACS>, m4<ERA5>, m4<IBTrACS>]

variable_combination: [[slp<ERA5>,slp<IBTrACS>,slp<forecast>], [wspeed<ERA5>,wspeed<IBTrACS>,wspeed<forecast>], [landfall_time<ERA5>,landfall_time<IBTrACS>,landfall_time<forecast>], [landfall_coordinate<ERA5>,landfall_coordinate<IBTrACS>,landfall_coordinate<forecast>], [formation_signal<IBTrACS>,formation_signal<forecast>]]  
  - same variable for obs and forecast; needs to be derived in forecast and ERA5
  - applied for all cases
  - derived through:  
        1. cyclone_dataset_builder.build_dataset  
        2. create_tctracks_from_dataset  
        3. find_landfall  


for each case:  
&emsp;  case <- from cases  
&emsp;&emsp;  for each applied_metric:  
&emsp;&emsp;&emsp;        metric, obs_type <- from applied_metrics  

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data
        5. compute variable_combination for forecast data
        6. compute variable_combination for observation data
        7. merge observation and forecast data
        6. compute applied_metric from observation and applied data

## Atmospheric River

### Refactor:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with atmospheric_river cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data (ERA5), convert into Dataset(s)  

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (specific humidity/relative humidity/dewpoint, temperature, u/v winds, geopotential_at_surface) and bounds  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**  
&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; for each variable in Observation:  
&emsp;&emsp;&emsp;&emsp; generate AR mask algorithm directed in Observation code  
&emsp;&emsp;&emsp;&emsp; combine subset forecast and Observation into EvaluationObject  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  

### Loopified:

event <- AR  
forecast <- AIFS

cases: [case1, case2] | event  
observation_types: [ERA5]  
metrics: [AR/Land Lead Time Detection (m1), AR/Land CoM Displacement Error (m2), AR/Land CSI (m3)]  
  - applied for all cases


applied_metrics <- cartesian product(metrics, observation_types) | iff observation type can be used by metric  
applied_metrics: [m1<ERA5>, m2<ERA5>, m3<ERA5>]

variable_combination: [ar/land intersection (v1)]  
  - same variable for obs and forecast
  - applied for all cases
  - derived through:  
        1. calculate_ivt  
        2. create_atmospheric_river_mask  
        3. find_land_intersection  


for each case:  
&emsp;  case <- from cases  
&emsp;&emsp;  for each applied_metric:  
&emsp;&emsp;&emsp;        metric, obs_type <- from applied_metrics  

        1. get event type information
        2. get metadata for the case
        3. load/harmonize forecast data for case
        4. load/harmonize observation data
        5. compute variable_combination for forecast data
        6. compute variable_combination for observation data
        7. merge observation and forecast data
        6. compute applied_metric from observation and applied data